# 🏷️ **PDF Page Importance Labeling Tool (Fixed)**

## **Purpose:**
Create a labeled dataset for training a CNN to classify pages as:
- ✅ **Important** (content pages)
- ❌ **Not Important** (cover, TOC, etc.)

## **Key Features:**
- 📸 Display **one page per PDF** at a time
- 🔄 **Resume sessions** - tracks which PDFs are completed
- ⏭️ **Skip Rest of PDF** - moves to next PDF without labeling remaining pages
- 📁 **Path-based tracking** - uses full file paths (handles duplicate names)
- 🚫 **No automatic labels** - only saves what you explicitly label
- 💾 **Auto-save** to Google Drive

---

## 🔧 **1. Setup & Installation**

In [1]:
# Install dependencies
!pip install -q PyMuPDF pillow ipywidgets

print("✅ Dependencies installed!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.1/24.1 MB 20.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 13.8 MB/s eta 0:00:00
✅ Dependencies installed!


## 📂 **2. Mount Google Drive & Configure Paths**

In [2]:
from google.colab import drive
import os
from pathlib import Path

# Mount Google Drive
if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')
    print("✅ Google Drive mounted!")
else:
    print("✅ Google Drive already mounted!")

# Configure paths
BASE_DIR = Path("/content/drive/Othercomputers/My Mac/Multi-lingual-Buddhist-Conversational-Agent/")
PDF_FOLDER = BASE_DIR / "data" / "raw" / "old_books"
PAGE_CLASSIFIER_DATA = BASE_DIR / "data" / "page_classifier_data"

# Create directories
IMAGES_DIR = PAGE_CLASSIFIER_DATA / "images"
LABELS_DIR = PAGE_CLASSIFIER_DATA / "labels"
PROGRESS_FILE = LABELS_DIR / "progress.json"  # NEW: Tracks which PDFs are done

for directory in [PAGE_CLASSIFIER_DATA, IMAGES_DIR, LABELS_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

# Create label subfolders
(IMAGES_DIR / "important").mkdir(exist_ok=True)
(IMAGES_DIR / "not_important").mkdir(exist_ok=True)

print(f"\n📂 Directory Structure:")
print(f"   PDFs: {PDF_FOLDER}")
print(f"   Images: {IMAGES_DIR}")
print(f"   Labels: {LABELS_DIR}")
print(f"   Progress: {PROGRESS_FILE}")

# Check if PDFs exist
pdf_files = list(PDF_FOLDER.rglob("*.pdf"))
print(f"\n📚 Found {len(pdf_files)} PDF files")

Mounted at /content/drive
✅ Google Drive mounted!

📂 Directory Structure:
   PDFs: /content/drive/Othercomputers/My Mac/Multi-lingual-Buddhist-Conversational-Agent/data/raw/old_books
   Images: /content/drive/Othercomputers/My Mac/Multi-lingual-Buddhist-Conversational-Agent/data/page_classifier_data/images
   Labels: /content/drive/Othercomputers/My Mac/Multi-lingual-Buddhist-Conversational-Agent/data/page_classifier_data/labels
   Progress: /content/drive/Othercomputers/My Mac/Multi-lingual-Buddhist-Conversational-Agent/data/page_classifier_data/labels/progress.json

📚 Found 28 PDF files


## 🎨 **3. Labeling Session Manager (REDESIGNED)**

In [3]:
import fitz  # PyMuPDF
from PIL import Image
import json
from datetime import datetime
from IPython.display import display, HTML, clear_output
import ipywidgets as widgets
from collections import defaultdict

# ═══════════════════════════════════════════════════════
# REDESIGNED LABELING SESSION
# ═══════════════════════════════════════════════════════

class LabelingSession:
    def __init__(self):
        self.pdf_files = sorted(list(PDF_FOLDER.rglob("*.pdf")))
        self.current_pdf_idx = 0
        self.current_page_idx = 0
        self.current_doc = None

        # NEW: Track progress per PDF file path
        self.progress = self.load_progress()

        # NEW: Track labeled images (not all pages, just what we save)
        self.labeled_images = self.load_labeled_images()

        # Find first unlabeled PDF
        self.find_next_unlabeled_pdf()

    def load_progress(self):
        """
        Load progress file which tracks which PDFs are completed
        Structure: {pdf_path: {status: 'complete'/'in_progress', last_page: int}}
        """
        if PROGRESS_FILE.exists():
            with open(PROGRESS_FILE, 'r') as f:
                progress = json.load(f)
            completed = sum(1 for p in progress.values() if p.get('status') == 'complete')
            print(f"✅ Loaded progress: {completed}/{len(self.pdf_files)} PDFs completed")
            return progress
        else:
            print("📝 Starting fresh labeling session")
            return {}

    def load_labeled_images(self):
        """
        Load information about already saved images
        Structure: {image_filename: {pdf_path: str, page_num: int, label: str}}
        """
        labels_file = LABELS_DIR / "labeled_images.json"
        if labels_file.exists():
            with open(labels_file, 'r') as f:
                labeled = json.load(f)
            print(f"✅ Loaded {len(labeled)} previously labeled images")
            return labeled
        else:
            return {}

    def save_progress(self):
        """Save progress file"""
        with open(PROGRESS_FILE, 'w') as f:
            json.dump(self.progress, f, indent=2)

    def save_labeled_images(self):
        """Save labeled images metadata"""
        labels_file = LABELS_DIR / "labeled_images.json"
        with open(labels_file, 'w') as f:
            json.dump(self.labeled_images, f, indent=2)
        print(f"💾 Saved {len(self.labeled_images)} labeled images")

    def find_next_unlabeled_pdf(self):
        """
        Find the first PDF that is not marked as complete
        """
        for idx, pdf_path in enumerate(self.pdf_files):
            pdf_key = str(pdf_path)
            if pdf_key not in self.progress or self.progress[pdf_key].get('status') != 'complete':
                self.current_pdf_idx = idx
                # If in progress, resume from last page
                if pdf_key in self.progress:
                    self.current_page_idx = self.progress[pdf_key].get('last_page', 0)
                else:
                    self.current_page_idx = 0
                return True

        # All PDFs completed
        print("🎉 All PDFs have been labeled!")
        return False

    def get_current_pdf_path(self):
        if self.current_pdf_idx < len(self.pdf_files):
            return str(self.pdf_files[self.current_pdf_idx])
        return None

    def get_current_pdf_name(self):
        if self.current_pdf_idx < len(self.pdf_files):
            return self.pdf_files[self.current_pdf_idx].stem
        return None

    def load_pdf(self, pdf_idx):
        """Load a PDF document"""
        if self.current_doc:
            self.current_doc.close()

        self.current_pdf_idx = pdf_idx
        pdf_path = self.pdf_files[pdf_idx]
        self.current_doc = fitz.open(pdf_path)

        # Check if resuming
        pdf_key = str(pdf_path)
        if pdf_key in self.progress:
            self.current_page_idx = self.progress[pdf_key].get('last_page', 0)
        else:
            self.current_page_idx = 0

    def get_page_image(self, page_num):
        """Extract page as PIL Image"""
        page = self.current_doc[page_num]
        pix = page.get_pixmap(dpi=150)
        img = Image.frombytes("RGB", [pix.width, pix.height], pix.samples)
        return img

    def get_next_image_number(self, label):
        """Get next available image number for a specific label (global counter)"""
        label_folder = IMAGES_DIR / label
        existing_files = list(label_folder.glob("page_*.png"))

        if existing_files:
            numbers = []
            for f in existing_files:
                try:
                    # Extract number from page_XXXX.png
                    num = int(f.stem.split('_')[1])
                    numbers.append(num)
                except:
                    pass
            print(f"Label: {label}\n Last Saved Page: {max(numbers) if numbers else 0}")
            return max(numbers) + 1 if numbers else 0
        return 0

    def save_page_image(self, page_num, label):
        """Save page image to label folder with global incremental numbering"""
        img = self.get_page_image(page_num)

        # Get next number for THIS label (independent counters)
        next_num = self.get_next_image_number(label)

        # Save image with simple incremental name
        label_folder = IMAGES_DIR / label
        img_filename = f"page_{next_num:04d}.png"
        img_path = label_folder / img_filename

        img.save(img_path)

        return str(img_path), img_filename

    def label_page(self, label):
        """Label current page (handles both new labels and re-labeling)"""
        pdf_path = self.get_current_pdf_path()
        page_num = self.current_page_idx

        # Check if this page was already labeled
        old_entry = self.find_existing_label(pdf_path, page_num)

        if old_entry:
            # RE-LABELING: Page was already labeled
            old_key, old_data = old_entry
            old_label = old_data['label']
            old_img_path = old_data.get('image_path')

            if old_label == label:
                # Same label - do nothing, just move to next page
                print(f"⚠️ Page {page_num} already labeled as '{label}'")
                return

            # Different label - need to move image and update JSON
            print(f"🔄 Re-labeling page {page_num}: {old_label} → {label}")

            # Delete old image file
            if old_img_path and Path(old_img_path).exists():
                Path(old_img_path).unlink()
                print(f"   🗑️ Deleted old image from {old_label}/")

            # Delete old JSON entry
            del self.labeled_images[old_key]

            # Save new image with new label
            img_path, img_filename = self.save_page_image(page_num, label)
            print(f"   ✅ Saved new image to {label}/")

        else:
            # NEW LABEL: Page never labeled before
            img_path, img_filename = self.save_page_image(page_num, label)

        # Create new JSON entry with unique key
        img_key = f"{label}/{img_filename}"

        self.labeled_images[img_key] = {
            'filename': img_filename,
            'pdf_path': pdf_path,
            'page_num': page_num,
            'label': label,
            'image_path': img_path,
            'timestamp': datetime.now().isoformat()
        }

        # Update progress
        if pdf_path not in self.progress:
            self.progress[pdf_path] = {}
        self.progress[pdf_path]['last_page'] = page_num
        self.progress[pdf_path]['status'] = 'in_progress'

        # Auto-save every 5 labels
        if len(self.labeled_images) % 5 == 0:
            self.save_labeled_images()
            self.save_progress()

    def find_existing_label(self, pdf_path, page_num):
        """
        Find if this page (from this PDF) was already labeled
        Returns: (key, data) tuple if found, None otherwise
        """
        for key, data in self.labeled_images.items():
            if (data.get('pdf_path') == pdf_path and
                data.get('page_num') == page_num):
                return (key, data)
        return None

    def skip_rest_of_pdf(self):
        """
        Mark current PDF as complete and move to next PDF
        Does NOT label remaining pages
        """
        pdf_path = self.get_current_pdf_path()

        # Mark as complete
        if pdf_path not in self.progress:
            self.progress[pdf_path] = {}
        self.progress[pdf_path]['status'] = 'complete'
        self.progress[pdf_path]['last_page'] = len(self.current_doc) - 1

        self.save_progress()
        self.save_labeled_images()

        return f"Marked PDF as complete (did not label remaining pages)"

    def get_stats(self):
        """Get labeling statistics"""
        important = sum(1 for img in self.labeled_images.values() if img['label'] == 'important')
        not_important = sum(1 for img in self.labeled_images.values() if img['label'] == 'not_important')

        completed_pdfs = sum(1 for p in self.progress.values() if p.get('status') == 'complete')

        return {
            'total_pdfs': len(self.pdf_files),
            'completed_pdfs': completed_pdfs,
            'current_pdf': self.current_pdf_idx + 1,
            'total_images_saved': len(self.labeled_images),
            'important': important,
            'not_important': not_important
        }

# Initialize session
session = LabelingSession()

if len(session.pdf_files) == 0:
    print("❌ No PDF files found! Please check the PDF_FOLDER path.")
else:
    print(f"\n✅ Labeling session initialized")
    stats = session.get_stats()
    print(f"   Total PDFs: {stats['total_pdfs']}")
    print(f"   Completed: {stats['completed_pdfs']}")
    print(f"   Images labeled: {stats['total_images_saved']}")

📝 Starting fresh labeling session

✅ Labeling session initialized
   Total PDFs: 28
   Completed: 0
   Images labeled: 0


## 🖼️ **4. Interactive Labeling Interface**

In [4]:
# ═══════════════════════════════════════════════════════
# INTERACTIVE WIDGET-BASED INTERFACE
# ═══════════════════════════════════════════════════════

# Output widgets
image_output = widgets.Output()
stats_output = widgets.Output()
info_output = widgets.Output()

# Buttons
btn_important = widgets.Button(
    description='✅ IMPORTANT (I)',
    button_style='success',
    layout=widgets.Layout(width='200px', height='50px')
)

btn_not_important = widgets.Button(
    description='❌ NOT IMPORTANT (N)',
    button_style='danger',
    layout=widgets.Layout(width='200px', height='50px')
)

btn_skip_pdf = widgets.Button(
    description='⏭️ SKIP REST OF PDF',
    button_style='warning',
    layout=widgets.Layout(width='200px', height='50px')
)

btn_prev = widgets.Button(
    description='← Previous',
    button_style='info',
    layout=widgets.Layout(width='150px')
)

btn_next_page = widgets.Button(
    description='Next Page →',
    button_style='info',
    layout=widgets.Layout(width='150px')
)

# ═══════════════════════════════════════════════════════
# DISPLAY FUNCTIONS
# ═══════════════════════════════════════════════════════

def update_display():
    """Update the page display"""
    if session.current_doc is None:
        return

    # Clear outputs
    image_output.clear_output(wait=True)
    info_output.clear_output(wait=True)
    stats_output.clear_output(wait=True)

    # Display current page info
    with info_output:
        pdf_name = session.get_current_pdf_name()
        pdf_path = session.get_current_pdf_path()
        total_pages = len(session.current_doc)
        page_num = session.current_page_idx

        # Check PDF status
        pdf_status = session.progress.get(pdf_path, {}).get('status', 'not started')
        status_color = 'green' if pdf_status == 'complete' else 'orange' if pdf_status == 'in_progress' else 'gray'

        # Check if this page is already labeled
        existing = session.find_existing_label(pdf_path, page_num)
        label_status = ""
        if existing:
            key, data = existing
            current_label = data['label']
            label_color = 'green' if current_label == 'important' else 'red'
            label_icon = '✅' if current_label == 'important' else '❌'
            label_status = f"<p style='color: {label_color}; font-weight: bold;'>{label_icon} Already labeled as: <b>{current_label.upper()}</b></p>"
            label_status += "<p style='font-size: 12px; color: #666;'>Click a button to re-label this page</p>"

        display(HTML(f"""
        <div style='background: #f0f0f0; padding: 15px; border-radius: 10px; margin-bottom: 10px;'>
            <h3>📄 {pdf_name}</h3>
            <p><b>Page:</b> {page_num + 1} / {total_pages} | <b>PDF:</b> {session.current_pdf_idx + 1} / {len(session.pdf_files)}</p>
            <p style='color: {status_color};'><b>Status:</b> {pdf_status.upper()}</p>
            {label_status}
            <p style='font-size: 11px; color: #666;'>{pdf_path}</p>
        </div>
        """))

    # Display page image (SMALLER SIZE)
    with image_output:
        img = session.get_page_image(session.current_page_idx)

        # Resize for display - CHANGED: max width from 800 to 600
        display_img = img.copy()
        max_width = 600  # Changed from 800 to 600

        if display_img.width > max_width:
            ratio = max_width / display_img.width
            new_height = int(display_img.height * ratio)
            display_img = display_img.resize((max_width, new_height))

        display(display_img)

    # Display statistics
    with stats_output:
        stats = session.get_stats()
        total = stats['important'] + stats['not_important']
        balance = abs(stats['important'] - stats['not_important'])

        balance_msg = "✅ Balanced" if balance <= 20 else f"⚠️ Imbalanced (+{balance})"

        display(HTML(f"""
        <div style='background: #e8f5e9; padding: 15px; border-radius: 10px;'>
            <h4>📊 Session Statistics</h4>
            <p><b>PDFs Completed:</b> {stats['completed_pdfs']} / {stats['total_pdfs']}</p>
            <p><b>Images Saved:</b> {stats['total_images_saved']}</p>
            <p>✅ <b>Important:</b> {stats['important']} | ❌ <b>Not Important:</b> {stats['not_important']}</p>
            <p><b>Balance:</b> {balance_msg}</p>
        </div>
        """))

# ═══════════════════════════════════════════════════════
# BUTTON CALLBACKS
# ═══════════════════════════════════════════════════════

def on_important_clicked(b):
    session.label_page('important')
    # Move to next page
    session.current_page_idx += 1
    if session.current_page_idx >= len(session.current_doc):
        # Auto-complete PDF and move to next
        session.skip_rest_of_pdf()
        if session.find_next_unlabeled_pdf():
            session.load_pdf(session.current_pdf_idx)
        else:
            with info_output:
                clear_output()
                print("🎉 All PDFs completed!")
            return
    update_display()

def on_not_important_clicked(b):
    session.label_page('not_important')
    # Move to next page
    session.current_page_idx += 1
    if session.current_page_idx >= len(session.current_doc):
        # Auto-complete PDF and move to next
        session.skip_rest_of_pdf()
        if session.find_next_unlabeled_pdf():
            session.load_pdf(session.current_pdf_idx)
        else:
            with info_output:
                clear_output()
                print("🎉 All PDFs completed!")
            return
    update_display()

def on_skip_pdf_clicked(b):
    message = session.skip_rest_of_pdf()
    with info_output:
        print(f"⏭️ {message}")

    # Move to next unlabeled PDF
    if session.find_next_unlabeled_pdf():
        session.load_pdf(session.current_pdf_idx)
        update_display()
    else:
        with info_output:
            clear_output()
            print("🎉 All PDFs completed!")

def on_prev_clicked(b):
    if session.current_page_idx > 0:
        session.current_page_idx -= 1
        update_display()

def on_next_page_clicked(b):
    session.current_page_idx += 1
    if session.current_page_idx >= len(session.current_doc):
        session.current_page_idx = len(session.current_doc) - 1
    update_display()

# Attach callbacks
btn_important.on_click(on_important_clicked)
btn_not_important.on_click(on_not_important_clicked)
btn_skip_pdf.on_click(on_skip_pdf_clicked)
btn_prev.on_click(on_prev_clicked)
btn_next_page.on_click(on_next_page_clicked)

# ═══════════════════════════════════════════════════════
# KEYBOARD SHORTCUTS
# ═══════════════════════════════════════════════════════

display(HTML("""
<script>
document.addEventListener('keydown', function(event) {
    if (event.key === 'i' || event.key === 'I') {
        document.querySelector('button[class*="success"]').click();
    } else if (event.key === 'n' || event.key === 'N') {
        document.querySelector('button[class*="danger"]').click();
    }
});
</script>
"""))

# Layout
button_row1 = widgets.HBox([btn_important, btn_not_important, btn_skip_pdf])
button_row2 = widgets.HBox([btn_prev, btn_next_page])

# Display interface
display(HTML("""
<div style='background: #fff3cd; padding: 15px; border-radius: 10px; margin-bottom: 15px;'>
    <h3>⌨️ Keyboard Shortcuts</h3>
    <p><b>I</b> = Important | <b>N</b> = Not Important</p>
    <p><b>Skip Rest of PDF</b> = Mark current PDF as done, move to next PDF</p>
</div>
"""))

display(stats_output)
display(info_output)
display(button_row1)
display(button_row2)
display(image_output)

# Load first unlabeled PDF and display
if len(session.pdf_files) > 0:
    if session.find_next_unlabeled_pdf():
        session.load_pdf(session.current_pdf_idx)
        update_display()
        print("\n🚀 Labeling tool ready! Use buttons or keyboard shortcuts (I/N)")
    else:
        print("\n🎉 All PDFs are already completed!")
else:
    print("❌ No PDFs found to label")

Output()

Output()

Output()


🚀 Labeling tool ready! Use buttons or keyboard shortcuts (I/N)
Label: not_important
 Last Saved Page: 0
Label: not_important
 Last Saved Page: 1
🔄 Re-labeling page 2: not_important → important
   🗑️ Deleted old image from not_important/
   ✅ Saved new image to important/
Label: important
 Last Saved Page: 0
🔄 Re-labeling page 3: important → not_important
   🗑️ Deleted old image from important/
Label: not_important
 Last Saved Page: 1
   ✅ Saved new image to not_important/
🔄 Re-labeling page 3: not_important → important
   🗑️ Deleted old image from not_important/
Label: important
 Last Saved Page: 0
   ✅ Saved new image to important/
🔄 Re-labeling page 3: important → not_important
   🗑️ Deleted old image from important/
Label: not_important
 Last Saved Page: 1
   ✅ Saved new image to not_important/
🔄 Re-labeling page 2: important → not_important
   🗑️ Deleted old image from important/
Label: not_important
 Last Saved Page: 2
   ✅ Saved new image to not_important/
🔄 Re-labeling page 3: 

## 📊 **5. View Statistics & Summary**

In [5]:
# ═══════════════════════════════════════════════════════
# DETAILED STATISTICS
# ═══════════════════════════════════════════════════════

import pandas as pd

def generate_detailed_summary():
    """Generate comprehensive summary"""

    stats = session.get_stats()

    print("="*80)
    print("📊 LABELING SESSION SUMMARY")
    print("="*80)

    print(f"\n📚 Total PDFs: {stats['total_pdfs']}")
    print(f"✅ Completed PDFs: {stats['completed_pdfs']}")
    print(f"🔄 In Progress: {stats['total_pdfs'] - stats['completed_pdfs']}")

    print(f"\n💾 Total Images Saved: {stats['total_images_saved']}")
    print(f"✅ Important: {stats['important']}")
    print(f"❌ Not Important: {stats['not_important']}")

    if stats['total_images_saved'] > 0:
        balance = abs(stats['important'] - stats['not_important'])
        print(f"\n⚖️ Class Balance: {balance} difference ({balance/stats['total_images_saved']*100:.1f}%)")

    # Show completed PDFs
    print(f"\n📋 Completed PDFs:")
    print("-"*80)
    completed = [pdf for pdf, info in session.progress.items() if info.get('status') == 'complete']
    for pdf in completed[:10]:  # Show first 10
        pdf_name = Path(pdf).stem
        print(f"   ✓ {pdf_name}")
    if len(completed) > 10:
        print(f"   ... and {len(completed) - 10} more")

    print("\n" + "="*80)
    print("💾 Data saved to:")
    print(f"   Images: {IMAGES_DIR}")
    print(f"   Progress: {PROGRESS_FILE}")
    print(f"   Labeled Images: {LABELS_DIR / 'labeled_images.json'}")
    print("="*80)

# Generate summary
generate_detailed_summary()

📊 LABELING SESSION SUMMARY

📚 Total PDFs: 28
✅ Completed PDFs: 0
🔄 In Progress: 28

💾 Total Images Saved: 0
✅ Important: 0
❌ Not Important: 0

📋 Completed PDFs:
--------------------------------------------------------------------------------

💾 Data saved to:
   Images: /content/drive/Othercomputers/My Mac/Multi-lingual-Buddhist-Conversational-Agent/data/page_classifier_data/images
   Progress: /content/drive/Othercomputers/My Mac/Multi-lingual-Buddhist-Conversational-Agent/data/page_classifier_data/labels/progress.json
   Labeled Images: /content/drive/Othercomputers/My Mac/Multi-lingual-Buddhist-Conversational-Agent/data/page_classifier_data/labels/labeled_images.json


## 💾 **6. Manual Save & Export**

In [6]:
# Manual save
session.save_labeled_images()
session.save_progress()
print("✅ All data manually saved to Google Drive!")

💾 Saved 0 labeled images
✅ All data manually saved to Google Drive!


## 🔄 **7. Reset Progress (Use Carefully!)**

In [7]:
# ⚠️ DANGER ZONE - Only run if you want to start fresh

# Uncomment to reset progress (keeps images, clears progress tracking)
# session.progress = {}
# session.save_progress()
# print("🔄 Progress reset! All PDFs marked as unlabeled.")
# print("⚠️ Images were NOT deleted - only progress tracking was reset.")

---

## 📚 **Usage Guide**

### **Key Changes from Original:**

1. **✅ One Page Per PDF Display**
   - Shows one page at a time from current PDF
   - Clear PDF name and page number
   
2. **✅ Proper Skip Functionality**
   - "Skip Rest of PDF" marks PDF as complete
   - Does NOT auto-label remaining pages
   - Moves to next unlabeled PDF

3. **✅ Path-Based Tracking**
   - Uses full file paths as keys
   - Handles PDFs with same name in different folders
   - Two files:
     - `progress.json` - tracks which PDFs are done
     - `labeled_images.json` - metadata for saved images

4. **✅ No Automatic Labels**
   - Only saves images you explicitly label
   - Skip = mark as done, no images saved

5. **✅ Resume Sessions**
   - Automatically finds first unlabeled PDF
   - Resumes from where you left off
   - Can label 5 PDFs, close, come back later

### **Workflow:**
```
1. Run all cells
2. See first unlabeled PDF, page 1
3. Press 'I' (important) or 'N' (not important)
4. Automatically shows next page
5. When done with PDF: Click "Skip Rest of PDF"
6. Automatically loads next unlabeled PDF
7. Close notebook anytime - progress saved
8. Next time: resumes from first unlabeled PDF
```

### **File Structure:**
```
page_classifier_data/
├── images/
│   ├── important/
│   │   ├── page_0000.png
│   │   ├── page_0001.png
│   │   └── ...
│   └── not_important/
│       ├── page_0000.png
│       └── ...
└── labels/
    ├── progress.json          # Which PDFs are done
    └── labeled_images.json    # Metadata for saved images
```

### **Progress File Format:**
```json
{
  "/path/to/book1.pdf": {
    "status": "complete",
    "last_page": 420
  },
  "/path/to/book2.pdf": {
    "status": "in_progress",
    "last_page": 15
  }
}
```

### **Labeled Images Format:**
```json
{
  "page_0000.png": {
    "pdf_path": "/path/to/book1.pdf",
    "page_num": 5,
    "label": "not_important",
    "timestamp": "2025-01-15T10:30:00"
  }
}
```

---

**Version:** 2.0 (Fixed)  
**Created for:** Buddhist PDF Page Classification Project  
**License:** MIT
